# 030 — Proposition-Based Chunking
**Chunking Series** | Dense X Retrieval (Chen 2023): split text into atomic propositions

Covers: Proposition extraction · Atomic fact indexing · Retrieval comparison vs standard chunks


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    return open(path, encoding='utf-8').read()

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
dl_text  = load_doc("deep_learning.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars")


✓ Loaded 22,486 chars


In [4]:
# ── 1. What is proposition-based chunking? ──────────────────────────────
print("STANDARD CHUNKING:")
print("  Split at N characters or sentence boundaries.")
print("  One chunk often contains MULTIPLE facts.")
print("  Retrieval returns a mix of relevant + irrelevant sentences.")
print()
print("PROPOSITION-BASED CHUNKING (Dense X Retrieval, Chen et al. 2023):")
print("  Use an LLM to decompose each chunk into ATOMIC PROPOSITIONS.")
print("  Each proposition = one self-contained verifiable fact.")
print("  Index each proposition as its own document.")
print("  Retrieval returns a single precise fact, not a multi-fact paragraph.")
print()
print("Example:")
print("  Chunk: 'BERT was created by Google in 2018. It uses a transformer")
print("          encoder and was pre-trained on Wikipedia and BooksCorpus.'")
print()
print("  Propositions:")
print("    P1: 'BERT was created by Google.'")
print("    P2: 'BERT was released in 2018.'")
print("    P3: 'BERT uses a transformer encoder architecture.'")
print("    P4: 'BERT was pre-trained on Wikipedia and BooksCorpus.'")


STANDARD CHUNKING:
  Split at N characters or sentence boundaries.
  One chunk often contains MULTIPLE facts.
  Retrieval returns a mix of relevant + irrelevant sentences.

PROPOSITION-BASED CHUNKING (Dense X Retrieval, Chen et al. 2023):
  Use an LLM to decompose each chunk into ATOMIC PROPOSITIONS.
  Each proposition = one self-contained verifiable fact.
  Index each proposition as its own document.
  Retrieval returns a single precise fact, not a multi-fact paragraph.

Example:
  Chunk: 'BERT was created by Google in 2018. It uses a transformer
          encoder and was pre-trained on Wikipedia and BooksCorpus.'

  Propositions:
    P1: 'BERT was created by Google.'
    P2: 'BERT was released in 2018.'
    P3: 'BERT uses a transformer encoder architecture.'
    P4: 'BERT was pre-trained on Wikipedia and BooksCorpus.'


In [5]:
# ── 2. Proposition extraction with LLM ──────────────────────────────────
PROP_PROMPT = (
    "Decompose the text below into atomic propositions.\n"
    "Rules:\n"
    "- Each proposition must be a single, self-contained factual statement\n"
    "- Each must be understandable without the original text (resolve pronouns)\n"
    "- Output one proposition per line\n"
    "- Do NOT add numbering, bullets, or extra text\n"
    "- Omit vague or uninformative statements\n\n"
    "Text:\n{text}\n\n"
    "Propositions:"
)

def extract_propositions(text: str) -> list:
    response = llm.invoke(PROP_PROMPT.format(text=text))
    lines = [l.strip() for l in response.content.strip().split("\n") if l.strip()]
    # Filter out non-proposition lines (too short or look like headers)
    return [l for l in lines if len(l) > 15 and not l.endswith(":")]

# Demo on a sample passage
sample = rag_text[:800]
props = extract_propositions(sample)
print(f"Input: {len(sample)} chars")
print(f"Extracted {len(props)} propositions:")
for i, p in enumerate(props[:10], 1):
    print(f"  {i}. {p}")


Input: 800 chars
Extracted 15 propositions:
  1. Retrieval-Augmented Generation (RAG) is a hybrid AI architecture.
  2. RAG combines information retrieval with text generation.
  3. RAG systems retrieve relevant documents from an external knowledge base.
  4. RAG systems use retrieved documents as context for generating responses.
  5. RAG was introduced by Lewis et al. in 2020.
  6. RAG addresses key limitations of parametric language models.
  7. Key limitations of parametric language models include knowledge staleness.
  8. Key limitations of parametric language models include hallucination.
  9. Key limitations of parametric language models include lack of attribution.
  10. A standard RAG pipeline consists of three stages.


In [6]:
# ── 3. Build proposition index from corpus ───────────────────────────────
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# First chunk, then extract propositions from each chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
# Use a subset for the demo (full corpus = many LLM calls)
demo_text = rag_text[:3000]
base_chunks = splitter.split_text(demo_text)

print(f"Base chunks: {len(base_chunks)}")
print("Extracting propositions...")

all_propositions = []
source_chunks = []
for i, chunk in enumerate(base_chunks):
    props = extract_propositions(chunk)
    for p in props:
        all_propositions.append(p)
        source_chunks.append(chunk)   # link proposition back to source
    print(f"  Chunk {i+1}/{len(base_chunks)}: {len(props)} propositions")
    time.sleep(0.2)

print(f"\n✓ Total propositions: {len(all_propositions)}")
print(f"  vs base chunks    : {len(base_chunks)}")
print(f"  avg props/chunk   : {len(all_propositions)/len(base_chunks):.1f}")


Base chunks: 8
Extracting propositions...


  Chunk 1/8: 6 propositions


  Chunk 2/8: 8 propositions


  Chunk 3/8: 9 propositions


  Chunk 4/8: 19 propositions


  Chunk 5/8: 12 propositions


  Chunk 6/8: 13 propositions


  Chunk 7/8: 12 propositions


  Chunk 8/8: 8 propositions

✓ Total propositions: 87
  vs base chunks    : 8
  avg props/chunk   : 10.9


In [7]:
# ── 4. Embed and index propositions ─────────────────────────────────────
import os, numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

os.environ["ANONYMIZED_TELEMETRY"] = "False"

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Build proposition store
prop_docs = [Document(page_content=p, metadata={"source_chunk": c[:80]})
             for p, c in zip(all_propositions, source_chunks)]
prop_store = FAISS.from_documents(prop_docs, embeddings)

# Build standard chunk store for comparison
chunk_docs = [Document(page_content=c) for c in base_chunks]
chunk_store = FAISS.from_documents(chunk_docs, embeddings)

print(f"✓ Proposition store: {prop_store.index.ntotal} vectors")
print(f"✓ Chunk store      : {chunk_store.index.ntotal} vectors")


✓ Proposition store: 87 vectors
✓ Chunk store      : 8 vectors


In [8]:
# ── 5. Retrieval comparison ─────────────────────────────────────────────
test_queries = [
    "What year was BERT released?",
    "How does RAG reduce hallucination?",
    "What is the purpose of retrieval in RAG?",
]

for query in test_queries:
    prop_results  = prop_store.similarity_search(query, k=3)
    chunk_results = chunk_store.similarity_search(query, k=3)

    print(f"\nQuery: {query!r}")
    print("  Proposition top-1:")
    print(f"    {prop_results[0].page_content!r}")
    print("  Standard chunk top-1:")
    print(f"    {chunk_results[0].page_content[:150]!r}")
    print(f"  Proposition is more specific: {len(prop_results[0].page_content) < len(chunk_results[0].page_content)}")



Query: 'What year was BERT released?'
  Proposition top-1:
    'BM25 excels at rare term retrieval.'
  Standard chunk top-1:
    'Retrieval-Augmented Generation (RAG)\n\nRetrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information\nretrieval with text g'
  Proposition is more specific: True

Query: 'How does RAG reduce hallucination?'
  Proposition top-1:
    'RAG addresses hallucination in language models.'
  Standard chunk top-1:
    'The quality of RAG depends heavily on how documents are chunked:\n- Fixed-size chunking: split at character boundaries; simple but ignores semantics\n- '
  Proposition is more specific: True

Query: 'What is the purpose of retrieval in RAG?'
  Proposition top-1:
    'RAG systems use retrieved documents as context for generating responses.'
  Standard chunk top-1:
    'Retrieval-Augmented Generation (RAG)\n\nRetrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information\nretrieval with text g'
  

In [9]:
# ── 6. Full proposition-based RAG pipeline ───────────────────────────────
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the atomic facts below.\n"
    "List which facts you used.\n\n"
    "Facts:\n{facts}\n\n"
    "Question: {question}\n\nAnswer:"
)

def proposition_rag(question: str, k: int = 6) -> str:
    retrieved = prop_store.similarity_search(question, k=k)
    facts = "\n".join([f"- {d.page_content}" for d in retrieved])
    chain = RAG_PROMPT | llm
    return chain.invoke({"facts": facts, "question": question}).content

answer = proposition_rag("How does retrieval augmented generation help LLMs?")
print(answer)


Retrieval-Augmented Generation (RAG) helps Large Language Models (LLMs) by combining information retrieval with text generation, allowing RAG systems to use retrieved documents as context for generating responses. This helps LLMs to leverage external knowledge and improve their performance.

Used facts:
1. RAG combines information retrieval with text generation.
2. RAG systems use retrieved documents as context for generating responses.


In [10]:
# ── 7. Summary ──────────────────────────────────────────────────────────
print("PROPOSITION CHUNKING — KEY POINTS")
print("=" * 45)
print()
print("Standard chunks contain multiple mixed facts.")
print("Propositions are atomic: one fact = one document.")
print()
print("Benefits:")
print("  Precise retrieval — no irrelevant sentences mixed in")
print("  Better for factoid questions: 'when?', 'who?', 'what year?'")
print("  Higher precision, fewer retrieved chunks needed")
print()
print("Costs:")
print("  1 LLM call per chunk at index time (expensive for large corpora)")
print("  More vectors to store (avg 5-8 props per chunk)")
print("  Slower to index than standard chunking")
print()
print("Best use cases:")
print("  Knowledge bases with factual content (encyclopedias, manuals)")
print("  Question-answering over dense technical documents")
print("  When retrieval precision matters more than recall")
print()
print("Paper: Dense X Retrieval (Chen et al., 2023)")
print("       'Propositions as atomic units for dense retrieval'")


PROPOSITION CHUNKING — KEY POINTS

Standard chunks contain multiple mixed facts.
Propositions are atomic: one fact = one document.

Benefits:
  Precise retrieval — no irrelevant sentences mixed in
  Better for factoid questions: 'when?', 'who?', 'what year?'
  Higher precision, fewer retrieved chunks needed

Costs:
  1 LLM call per chunk at index time (expensive for large corpora)
  More vectors to store (avg 5-8 props per chunk)
  Slower to index than standard chunking

Best use cases:
  Knowledge bases with factual content (encyclopedias, manuals)
  Question-answering over dense technical documents
  When retrieval precision matters more than recall

Paper: Dense X Retrieval (Chen et al., 2023)
       'Propositions as atomic units for dense retrieval'
